Universal filter function and Loading different kinds of data

Loading diff data types
1. CSV. file
2. TXT FILE
3. WESAD PICKLE FILE

ACCESSING DIFFERENT SIGNALS
1. wrist eda
2. chest eda


# Step 1
Load data

# Step 2
Extract signal

eda_raw = wrist['EDA']

# Step 3
Visualize

plt.plot(eda_raw)

# Step 4
Filter

eda_filtered = lowpass_filter(
    eda_raw,
    cutoff=1,
    fs=4
)

# Step 5
Visualize again

plt.plot(eda_filtered)

# Step 6
Compare metrics

mean
std
noise
spikes
drift

# 1. Load data
df = ...

# 2. Filtering
ecg_filtered = butter_filter(ecg_raw)

# 3. Normalization
ecg_norm = normalize_signal(ecg_filtered)

# 4. Windowing
ecg_windows = create_overlapping_windows(ecg_norm, fs=700, window_size_sec=5, overlap=0.5)

In [ ]:
#CSV 
import pandas as pd

data = pd.read_csv("eda.csv")

#TXT
import pandas as pd

data = pd.read_csv(
    "file.txt",
    sep="\t",
    header=None
)

#PICKLE
import pickle

with open("S2.pkl", "rb") as file:
    data = pickle.load(file, encoding="latin1")

FileNotFoundError: [Errno 2] No such file or directory: 'eda.csv'

In [ ]:
#ACCESSING DATA
# pickle file
#1. 
wrist = data['signal']['wrist']
print(wrist.keys())
#OUTPUT
# dict_keys([
# 'ACC',
# 'BVP',
# 'EDA',
# 'TEMP'
# ])

eda_raw = wrist['EDA']

#2. 
chest = data['signal']['chest']
print(chest.keys())
# OUTPUT
# dict_keys([
# 'ACC',
# 'ECG',
# 'EDA',
# 'EMG',
# 'TEMP',
# 'Resp'
# ])
eda_raw = chest['EDA']

# csv
# 3. 
data = pd.read_csv("eda_data.csv", header=None)
# index0= start time, index 1= sampling frequency, index 2= EDA data
eda_raw = data['EDA'].iloc[2:].astype(float).values  

print(data.head())
print(data.columns)

eda_raw = data['EDA']

In [ ]:
# VISUALIZATION
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(eda_raw)
plt.xlabel("Samples")
plt.ylabel("EDA/temp/etc")
plt.title("Raw EDA")
plt.show()

Universal filter function and calling different filters

In [ ]:
def apply_filter(signal,
                 filter_type,
                 fs,
                 cutoff,
                 order=4):

In [ ]:
eda = apply_filter(
    eda_raw,
    'lowpass',
    fs=4,
    cutoff=1
)

ecg = apply_filter(
    ecg_raw,
    'bandpass',
    fs=700,
    cutoff=[0.5,40]
)

COMPARE METRICS
1. COMPARE GRAPHS
2. COMPARE SMALL PLOT OF A GRPAH
3. COMPARING METRICS


just define signal_raw & signal_filtered

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Compare Length
# -------------------------

print("Raw Samples      :", len(signal_raw))
print("Filtered Samples :", len(signal_filtered))

# -------------------------
# Compare Statistics
# -------------------------

print("\n===== RAW SIGNAL =====")
print("Mean :", np.mean(signal_raw))
print("Std  :", np.std(signal_raw))

print("\n===== FILTERED SIGNAL =====")
print("Mean :", np.mean(signal_filtered))
print("Std  :", np.std(signal_filtered))

# -------------------------
# Plot Comparison
# -------------------------

plt.figure(figsize=(12,4))

plt.plot(signal_raw,
         label='Raw Signal',
         alpha=0.6)

plt.plot(signal_filtered,
         label='Filtered Signal',
         linewidth=2)

plt.title("Raw vs Filtered Signal")
plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.legend()
plt.grid(True)

plt.show()

# compare small portion of graph
start = 0
end = 1000

plt.figure(figsize=(12,4))

plt.plot(signal_raw[start:end],
         label='Raw')

plt.plot(signal_filtered[start:end],
         label='Filtered')

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from scipy.signal import find_peaks
from scipy.stats import linregress

NOISE LEVEL

In [ ]:
def calculate_noise(signal):
    return np.std(np.diff(signal))
noise = calculate_noise(eda_raw)
print("Noise:", noise)

SPIKE COUNT

In [ ]:
def count_spikes(signal, threshold=3):
    
    diff_signal = np.abs(np.diff(signal))
    
    spike_threshold = threshold * np.std(diff_signal)
    
    spikes = np.sum(diff_signal > spike_threshold)
    
    return spikes
spikes = count_spikes(eda_raw)
print("Spikes:", spikes)

PEAK COUNT

In [ ]:
def count_peaks(signal):
    
    peaks, _ = find_peaks(signal)
    
    return len(peaks)
peaks = count_peaks(eda_raw)
print("Peaks:", peaks)
# EDA
peaks, _ = find_peaks(signal, prominence=0.02)

DRIFT SLOPE

In [1]:
def calculate_drift(signal):
    
    x = np.arange(len(signal))
    
    slope, _, _, _, _ = linregress(x, signal)
    
    return slope
drift = calculate_drift(eda_raw)
print("Drift:", drift)

NameError: name 'eda_raw' is not defined

In [ ]:
print("Noise      :", calculate_noise(signal))
print("Spikes     :", count_spikes(signal))
print("Peaks      :", count_peaks(signal))
print("Drift Slope:", calculate_drift(signal))

GENERIC CODE

In [ ]:
import numpy as np
from scipy.stats import linregress

# -------------------------
# Noise
# -------------------------
def calculate_noise(signal):
    return np.std(np.diff(signal))

# -------------------------
# Spikes
# -------------------------
def count_spikes(signal, threshold=3):
    diff_signal = np.abs(np.diff(signal))
    spike_threshold = threshold * np.std(diff_signal)
    return np.sum(diff_signal > spike_threshold)

# -------------------------
# Drift
# -------------------------
def calculate_drift(signal):
    x = np.arange(len(signal))
    slope, _, _, _, _ = linregress(x, signal)
    return slope

# -------------------------
# Generic Metrics
# -------------------------
def get_metrics(signal):
    return {
        "Samples": len(signal),
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Min": np.min(signal),
        "Max": np.max(signal),
        "Range": np.max(signal) - np.min(signal),
        "Noise": calculate_noise(signal),
        "Spikes": count_spikes(signal),
        "Drift": calculate_drift(signal)
    }

# -------------------------
# Compare Raw vs Filtered
# -------------------------
def compare_signals(raw_signal, filtered_signal, signal_name="Signal"):

    raw_metrics = get_metrics(raw_signal)
    filtered_metrics = get_metrics(filtered_signal)

    print(f"\n========== {signal_name.upper()} ==========")

    for key in raw_metrics.keys():
        print(
            f"{key:<10} | Raw: {raw_metrics[key]:<12.6f} | Filtered: {filtered_metrics[key]:<12.6f}"
        )

    return raw_metrics, filtered_metrics

In [ ]:
raw_metrics, filtered_metrics = compare_signals(
    temp_celsius,
    filtered_temp,
    "Temperature"
)

print("\n========== CHANGE ==========")

for key in raw_metrics.keys():

    raw = raw_metrics[key]
    filt = filtered_metrics[key]

    if raw != 0:
        change = ((filt - raw) / raw) * 100
        print(f"{key:<10}: {change:.2f}%")